# 01 · Ingest, validate, chunk and retain provenance

> **Core lesson:** all cells execute without external services. Read the *predict → run → explain* questions before each experiment.


## The data engineering question

**A document is not automatically a good retrieval unit.** Policies may contain several independent facts. Chunks are indexed so that the most relevant section can be retrieved.

Our JSONL document contract is `doc_id`, `title`, `text`, `source`. Each chunk additionally preserves `start_word` and `end_word` (Python half-open range), enabling reproducible traceability.

**Predict:** Lowering chunk size from 80 to 12 words usually yields **more chunks**, but a sentence can be split across boundaries. Overlap recovers some neighbor context at the cost of duplication.


In [ ]:
from pathlib import Path
from RAG.src.core import load_documents, chunk_documents
docs = load_documents(Path("RAG/data/support_policies.jsonl"))
for size, overlap in [(80, 15), (24, 6), (12, 3)]:
    cs = chunk_documents(docs, size=size, overlap=overlap)
    print(f"size={size:>2}, overlap={overlap:>2} -> {len(cs):>2} chunks")


## Trace one policy through multiple overlapping chunks

The original `doc_id` persists, but chunk IDs are unique to positions. Overlap is counted in **words**, not tokens from a neural tokenizer: token budgets need model-specific tokenization in production.


In [ ]:
short = chunk_documents([docs[0]], size=12, overlap=3)
for c in short[:3]:
    print(f"{c.chunk_id} | words [{c.start_word}:{c.end_word}] | {c.text}")
if len(short) > 1:
    assert short[0].text.split()[-3:] == short[1].text.split()[:3]


## Validate inputs instead of silently repairing errors

Duplicate IDs can contaminate citations and evaluation. Do **not** index content simply because a text extractor returned a string; validate provenance, permissions and parsing quality.


In [ ]:
from RAG.src.core import Document
try:
    chunk_documents(docs, size=12, overlap=12)
except ValueError as error:
    print("Expected validation:", error)
example = Document("demo", "Boundary", "one two three four five six seven", "classroom")
for c in chunk_documents([example], size=4, overlap=2):
    print(c.chunk_id, c.text)


## Design exercise

Draw a policy table (refund type, amount, deadline). What breaks if rows are flattened into unrelated text chunks? Propose a table-aware representation.

**Change → effect:** set `overlap=0` and compare whether a complete evidence sentence can be reconstructed from any *single* chunk. More overlap can improve coverage but also reduce unique information in top-k.

**Success criterion:** for any answer, identify the exact document and word span that supplied its evidence. Continue to **02_retrieve_and_rank**.
